# PTCG 0034 merged recent-three-day cleaned BC

Trains three complete epochs over the merged cleaned train split from 2026-08-01 through 2026-08-03; each epoch includes all three days and keeps d_model=320, four state layers, and three option layers.

## Environment and immutable training contract

In [ ]:

from __future__ import annotations

import hashlib
import importlib
import json
import math
import os
from pathlib import Path
import random
import shutil
import sys
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

INPUT = Path(os.environ.get("PTCG_KAGGLE_INPUT", "/kaggle/input"))
WORKING = Path(os.environ.get("PTCG_KAGGLE_WORKING", "/kaggle/working"))
OUTPUT = WORKING / "ptcg_0034_recent3_cleaned_bc_train"
SMOKE = os.environ.get("PTCG_NOTEBOOK_SMOKE") == "1"
SEED = 20260804
EPOCHS = 3
TRAIN_BATCH_SIZE = 512 if SMOKE else 192
VALIDATION_BATCH_SIZE = 512 if SMOKE else 256
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.02
PREFETCH_DEPTH = 2
EXPECTED_PARAMETER_COUNT = 22_595_202
EXPECTED_DATES = {
        '2026-08-01',
        '2026-08-02',
        '2026-08-03',
    }

if EPOCHS < 1:
    raise ValueError("0034 merged training requires at least one complete epoch")
if not torch.cuda.is_available():
    raise RuntimeError("0034 training requires CUDA")
gpu_count = torch.cuda.device_count()
device_names = [torch.cuda.get_device_name(index) for index in range(gpu_count)]
if SMOKE:
    device_ids = list(range(min(2, gpu_count)))
else:
    if gpu_count < 2 or any("T4" not in name for name in device_names[:2]):
        raise RuntimeError(f"0034 formal training requires two T4 GPUs: {device_names}")
    device_ids = [0, 1]
if not device_ids:
    raise RuntimeError("no CUDA device selected")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_json(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as handle:
        json.dump(
            payload,
            handle,
            ensure_ascii=True,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)


def _source_schema(root: Path) -> str | None:
    fields_path = root / "contracts" / "fields.py"
    if not fields_path.is_file():
        return None
    for line in fields_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("SCHEMA_VERSION") and "=" in line:
            return line.split("=", 1)[1].strip().strip(chr(34)).strip(chr(39))
    return None


def _is_0034_source(root: Path) -> bool:
    return _source_schema(root) == "0034_clean_recent3_semantic_decision_v1"


def _stage_source(root: Path, package_name: str) -> Path:
    staged = WORKING / package_name
    if staged.exists():
        shutil.rmtree(staged)
    shutil.copytree(root, staged)
    fields_path = staged / "contracts" / "fields.py"
    fields_text = fields_path.read_text(encoding="utf-8")
    fields_text = fields_text.replace(
        "0033_effect_summary_semantic_decision_v1",
        "0034_clean_recent3_semantic_decision_v1",
    )
    fields_path.write_text(fields_text, encoding="utf-8")
    feature_audit_path = staged / "contracts" / "feature_audit.json"
    if feature_audit_path.is_file():
        feature_audit = json.loads(feature_audit_path.read_text(encoding="utf-8"))
        feature_audit["actor_schema"] = "0034_clean_recent3_semantic_decision_v1"
        feature_audit_path.write_text(json.dumps(feature_audit, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if _source_schema(staged) != "0034_clean_recent3_semantic_decision_v1":
        raise ValueError(f"staged source has unexpected schema: {_source_schema(staged)}")
    return staged


def find_source_root() -> Path:
    package_name = "0034_clean_recent3_semantic_bc"
    package_candidates = []
    flat_candidates = []
    rejected = []
    for public_asset in INPUT.rglob("official_public_prototypes_v1.json"):
        root = public_asset.parent.parent
        if not (
            (root / "features" / "audit.py").is_file()
            and (root / "training" / "dataset.py").is_file()
        ):
            continue
        schema = _source_schema(root)
        if schema not in {
            "0034_clean_recent3_semantic_decision_v1",
            "0033_effect_summary_semantic_decision_v1",
        }:
            rejected.append({"root": str(root), "schema": schema})
            continue
        if root.name == package_name:
            package_candidates.append(root)
        else:
            flat_candidates.append(root)
    package_unique = sorted(set(package_candidates))
    if len(package_unique) == 1:
        return _stage_source(package_unique[0], package_name)
    flat_unique = sorted(set(flat_candidates))
    if len(flat_unique) == 1:
        return _stage_source(flat_unique[0], package_name)
    raise FileNotFoundError(
        "expected one complete 0034 source root or one flat 0034 source dataset, "
        f"found package={package_unique}, flat={flat_unique}, rejected={rejected}"
    )


SOURCE_ROOT = find_source_root()
sys.path.insert(0, str(SOURCE_ROOT.parent))
PACKAGE = SOURCE_ROOT.name
fields = importlib.import_module(f"{PACKAGE}.contracts.fields")
batch_contract = importlib.import_module(f"{PACKAGE}.contracts.batch")
dataset_module = importlib.import_module(f"{PACKAGE}.training.dataset")
prefetch_module = importlib.import_module(f"{PACKAGE}.training.prefetch")
prototypes_module = importlib.import_module(f"{PACKAGE}.domain.prototypes")
model_module = importlib.import_module(f"{PACKAGE}.model")

if not hasattr(dataset_module, "CombinedCanonicalDecisionDataset"):
    raise RuntimeError("0034 source does not contain the combined partition loader")
if fields.SCHEMA_VERSION != "0034_clean_recent3_semantic_decision_v1":
    raise ValueError(f"unexpected actor schema: {fields.SCHEMA_VERSION}")

print(
    json.dumps(
        {
            "event": "0034_cleaned_training_environment",
            "smoke": SMOKE,
            "epochs": EPOCHS,
            "gpu_count": gpu_count,
            "device_ids": device_ids,
            "device_names": device_names,
            "wandb": None,
        },
        sort_keys=True,
    ),
    flush=True,
)


## Three-day dataset acceptance

In [ ]:

daily = {}
for manifest_path in INPUT.rglob("manifest.json"):
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if manifest.get("schema_version") != "0034_daily_semantic_partition_v1":
        continue
    date = manifest.get("date")
    if date not in EXPECTED_DATES:
        continue
    if date in daily:
        raise RuntimeError(f"duplicate daily partition attached for {date}")
    if manifest.get("status") != "complete" or manifest.get("winner_only") is not True:
        raise ValueError(f"daily partition is not a complete winner-only dataset: {date}")
    counts = manifest.get("split_counts", {})
    if any(int(counts.get(split, 0)) <= 0 for split in ("train", "validation")):
        raise ValueError(f"daily partition contains an empty split: {date}")
    feature_audit = manifest.get("feature_input_audit", {})
    acceptance = manifest.get("acceptance", {})
    if (
        feature_audit.get("status") != "passed"
        or feature_audit.get("audited_decisions") != manifest.get("records")
        or acceptance.get("status") != "passed"
        or acceptance.get("validated_records") != manifest.get("records")
        or acceptance.get("model_forward", {}).get("parameter_count")
        != EXPECTED_PARAMETER_COUNT
    ):
        raise ValueError(f"daily feature/model acceptance is incomplete: {date}")
    canonical_root = manifest_path.parent / manifest.get("canonical_path", "canonical")
    canonical_manifest_path = canonical_root / "manifest.json"
    if (
        not canonical_manifest_path.is_file()
        or sha256_file(canonical_manifest_path)
        != manifest.get("canonical_manifest_sha256")
    ):
        raise ValueError(f"canonical manifest commitment mismatch: {date}")
    daily[date] = {
        "manifest_path": manifest_path,
        "manifest": manifest,
        "canonical_root": canonical_root,
        "canonical_manifest_sha256": manifest["canonical_manifest_sha256"],
    }

if set(daily) != EXPECTED_DATES:
    raise FileNotFoundError(
        f"expected all three daily datasets; missing={sorted(EXPECTED_DATES - set(daily))}"
    )
canonical_hashes = [daily[date]["canonical_manifest_sha256"] for date in sorted(daily)]
if not SMOKE and len(set(canonical_hashes)) != len(canonical_hashes):
    raise ValueError("formal daily partitions contain duplicate canonical manifests")

roots = [daily[date]["canonical_root"] for date in sorted(daily)]
dataset = dataset_module.CombinedCanonicalDecisionDataset(roots)
expected_counts = {
    split: sum(int(daily[date]["manifest"]["split_counts"][split]) for date in daily)
    for split in ("train", "validation")
}
if dataset.split_counts != expected_counts:
    raise ValueError("combined dataset split counts disagree with daily manifests")
if dataset.manifest.get("partition_count") != 3:
    raise ValueError("combined dataset did not retain all three daily partitions")

inventory = [
    {
        "date": date,
        "records": daily[date]["manifest"]["records"],
        "split_counts": daily[date]["manifest"]["split_counts"],
        "canonical_manifest_sha256": daily[date]["canonical_manifest_sha256"],
    }
    for date in sorted(daily)
]
atomic_json(
    OUTPUT / "dataset_reference.json",
    {
        "schema_version": "0034_combined_dataset_reference_v1",
        "winner_only": True,
        "dates": sorted(EXPECTED_DATES),
        "combined_manifest_sha256": dataset.manifest_sha256,
        "combined_manifest": dataset.manifest,
        "inventory": inventory,
    },
)
print(
    json.dumps(
        {
            "event": "0034_dataset_preflight",
            "dates": sorted(EXPECTED_DATES),
            "partitions": len(roots),
            "split_counts": dataset.split_counts,
            "combined_manifest_sha256": dataset.manifest_sha256,
        },
        sort_keys=True,
    ),
    flush=True,
)


daily_partitions = [
    {
        "date": date,
        "dataset": dataset_module.CanonicalDecisionDataset(daily[date]["canonical_root"]),
        "split_counts": daily[date]["manifest"]["split_counts"],
        "canonical_manifest_sha256": daily[date]["canonical_manifest_sha256"],
    }
    for date in sorted(daily)
]
if sum(item["dataset"].split_counts["train"] for item in daily_partitions) != dataset.split_counts["train"]:
    raise ValueError("rolling daily train counts do not sum to the combined train split")
if sum(item["dataset"].split_counts["validation"] for item in daily_partitions) != dataset.split_counts["validation"]:
    raise ValueError("rolling daily validation counts do not sum to the combined validation split")
print(
    json.dumps(
        {
            "event": "0034_merged_partition_inventory",
            "stages": [
                {"date": item["date"], "split_counts": item["split_counts"]}
                for item in daily_partitions
            ],
        },
        sort_keys=True,
    ),
    flush=True,
)


## Exact model and dual-GPU preflight

In [ ]:

PROTOTYPE_PATH = SOURCE_ROOT / "assets" / "official_public_prototypes_v1.json"
prototypes = prototypes_module.PrototypeIndex.load(PROTOTYPE_PATH)
prototype_counts = {
    "cards": len(prototypes.engine_cards),
    "attacks": len(prototypes.engine_attacks),
    "skills": len(prototypes.skills),
}
if prototype_counts != {"cards": 1267, "attacks": 1556, "skills": 433}:
    raise ValueError(f"incomplete full-engine prototypes: {prototype_counts}")

model_config = model_module.ModelConfig(
    d_model=320,
    state_layers=4,
    option_layers=3,
    ffn_multiplier=3,
)
policy = model_module.SemanticPolicy(model_config, prototypes)
parameter_count = sum(parameter.numel() for parameter in policy.parameters())
if parameter_count != EXPECTED_PARAMETER_COUNT:
    raise ValueError(f"0034 parameter count drift: {parameter_count}")


class TeacherLogits(nn.Module):
    def __init__(self, inner: nn.Module):
        super().__init__()
        self.policy = inner

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        return self.policy.teacher_logits(batch)


primary = torch.device("cuda:0")
parallel = nn.DataParallel(
    TeacherLogits(policy).to(primary),
    device_ids=device_ids,
    output_device=device_ids[0],
)
optimizer = torch.optim.AdamW(
    parallel.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
    init_scale=4096.0,
    growth_interval=10_000,
)
amp_dtype = torch.float16

sample = next(dataset.iter_batches("train", TRAIN_BATCH_SIZE, seed=SEED))
batch_contract.DecisionBatch.from_mapping(sample)
smoke_input = {
    name: value[:8].detach().cpu()
    for name, value in sample.items()
}
torch.save(smoke_input, OUTPUT / "smoke_input.pt")
sample = {name: value.to(primary) for name, value in sample.items()}
parallel.eval()
with torch.inference_mode(), torch.autocast("cuda", dtype=amp_dtype):
    sample_logits = parallel(sample)
if sample_logits.ndim != 3 or not torch.isfinite(sample_logits).all():
    raise ValueError("dual-device teacher-logit preflight failed")
del sample, sample_logits, smoke_input
torch.cuda.empty_cache()

model_contract = {
    "schema_version": "0034_audited_model_contract_v1",
    "package_name": PACKAGE,
    "architecture": "SemanticPolicy",
    "model_config": model_config.to_dict(),
    "parameter_count": parameter_count,
    "actor_schema": fields.SCHEMA_VERSION,
    "action_contract": "ordered_legal_option_pointer_plus_stop",
    "initialized_from_checkpoint": None,
    "random_initialization": True,
    "data_parallel_device_ids": device_ids,
    "gpu_device_names": device_names,
    "winner_only": True,
    "wandb": None,
    "checkpoint_state_dict": "trainable_parameters_only",
    "prototype_assets": ["official_public_prototypes_v1.json", "official_full_engine_prototypes_v2.json"],
    "sample_weight_field": "audit.cleaning.sample_weight",
}
atomic_json(OUTPUT / "model_contract.json", model_contract)
shutil.copy2(PROTOTYPE_PATH, OUTPUT / PROTOTYPE_PATH.name)
shutil.copy2(
    PROTOTYPE_PATH.with_name("official_full_engine_prototypes_v2.json"),
    OUTPUT / "official_full_engine_prototypes_v2.json",
)
print(
    json.dumps(
        {
            "event": "0034_model_preflight",
            "parameter_count": parameter_count,
            "model_config": model_config.to_dict(),
            "initialized_from_checkpoint": None,
            "random_initialization": True,
        },
        sort_keys=True,
    ),
    flush=True,
)

for asset in ("official_public_prototypes_v1.json", "official_full_engine_prototypes_v2.json"):
    source_asset = PROTOTYPE_PATH if asset.startswith("official_public") else PROTOTYPE_PATH.with_name(asset)
    shutil.copy2(source_asset, OUTPUT / asset)
shutil.copy2(SOURCE_ROOT / "contracts" / "feature_audit.json", OUTPUT / "feature_audit.json")
model_source = OUTPUT / "model_source" / PACKAGE
model_source.mkdir(parents=True, exist_ok=True)
for source_name in ("__init__.py", "contracts", "domain", "features", "knowledge", "model"):
    source_path = SOURCE_ROOT / source_name
    target_path = model_source / source_name
    if source_path.is_dir():
        shutil.copytree(source_path, target_path)
    elif source_path.is_file():
        shutil.copy2(source_path, target_path)
atomic_json(OUTPUT / "model_package_manifest.json", {
    "schema_version": "0034_model_package_v1",
    "package_name": PACKAGE,
    "weights": ["best_model.pt", "last_model.pt"],
    "contract": "model_contract.json",
    "prototype_assets": ["official_public_prototypes_v1.json", "official_full_engine_prototypes_v2.json"],
    "feature_audit": "feature_audit.json",
    "cleaning_report": "cleaning_report.json",
    "training_report": "training_report.json",
    "training_config": "training_config.json",
    "dataset_reference": "dataset_reference.json",
    "usage_loader": "load_model.py",
    "usage_readme": "MODEL_USAGE.md",
    "smoke_input": "smoke_input.pt",
})
(OUTPUT / "load_model.py").write_text("\"\"\"Load the 0034 cleaned BC model package without hidden training state.\"\"\"\n\nfrom __future__ import annotations\n\nimport importlib\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\n\n\ndef load_model(package_dir: str | Path, *, device: str | torch.device = \"cpu\"):\n    package_dir = Path(package_dir)\n    contract = json.loads((package_dir / \"model_contract.json\").read_text(encoding=\"utf-8\"))\n    source_root = package_dir / \"model_source\"\n    if not source_root.is_dir():\n        raise FileNotFoundError(\"model package is missing model_source\")\n    sys.path.insert(0, str(source_root))\n    package = contract.get(\"package_name\", \"0034_clean_recent3_semantic_bc\")\n    domain = importlib.import_module(f\"{package}.domain.prototypes\")\n    config_module = importlib.import_module(f\"{package}.model.config\")\n    policy_module = importlib.import_module(f\"{package}.model.policy\")\n    prototypes = domain.PrototypeIndex.load(\n        package_dir / \"official_public_prototypes_v1.json\",\n        package_dir / \"official_full_engine_prototypes_v2.json\",\n    )\n    config = config_module.ModelConfig(**contract[\"model_config\"])\n    policy = policy_module.SemanticPolicy(config, prototypes).to(device).eval()\n    try:\n        payload = torch.load(package_dir / \"best_model.pt\", map_location=device, weights_only=True)\n    except TypeError:\n        payload = torch.load(package_dir / \"best_model.pt\", map_location=device)\n    if set(payload) != {\"schema_version\", \"state_dict\", \"metadata\"}:\n        raise ValueError(\"checkpoint payload must contain only schema_version/state_dict/metadata\")\n    state = payload[\"state_dict\"]\n    if any(str(key).startswith(\"module.\") for key in state):\n        raise ValueError(\"checkpoint contains forbidden DataParallel module. prefix\")\n    missing, unexpected = policy.load_state_dict(state, strict=False)\n    unexpected = list(unexpected)\n    parameter_names = {name for name, _ in policy.named_parameters()}\n    allowed_missing_prefixes = (\n        \"state_encoder.prototypes.\",\n        \"option_encoder.prototypes.\",\n        \"prototype_encoder.\",\n    )\n    invalid_missing = [\n        name for name in missing\n        if not str(name).startswith(allowed_missing_prefixes)\n    ]\n    if unexpected or invalid_missing:\n        raise ValueError({\"unexpected\": unexpected, \"invalid_missing\": invalid_missing})\n    return policy, contract\n", encoding="utf-8")
(OUTPUT / "MODEL_USAGE.md").write_text(
    "# 0034 cleaned BC model package\n\n"
    "Use `load_model.py` with this directory. The contract and both prototype assets are required; "
    "the checkpoint contains trainable parameters only and reconstructs deterministic prototype buffers. "
    "Load `smoke_input.pt` with `torch.load(..., weights_only=True)` to run a package-only finite forward.\n",
    encoding="utf-8",
)
aggregate_cleaning_reports = []
for cleaning_report in INPUT.rglob("cleaning_report.json"):
    try:
        cleaning_payload = json.loads(cleaning_report.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if (
        cleaning_payload.get("schema_version") == "0034_cleaning_report_v1"
        and set(cleaning_payload.get("dates", [])) == EXPECTED_DATES
        and cleaning_payload.get("status") == "passed"
    ):
        aggregate_cleaning_reports.append(cleaning_report)
if len(aggregate_cleaning_reports) != 1:
    raise RuntimeError(
        f"expected exactly one aggregate 0034 cleaning report, found {aggregate_cleaning_reports}"
    )
shutil.copy2(aggregate_cleaning_reports[0], OUTPUT / "cleaning_report.json")


## Weighted merged training and validation functions

In [ ]:

def move_batch(batch: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    return {
        name: value.to(primary, non_blocking=True)
        for name, value in batch.items()
    }

def _audit_sample_weight(audit: dict) -> float:
    cleaning = audit.get("cleaning", {}) if isinstance(audit, dict) else {}
    value = cleaning.get("sample_weight", 1.0)
    try:
        value = float(value)
    except (TypeError, ValueError):
        value = 1.0
    if not math.isfinite(value) or value <= 0:
        return 1.0
    return value


def _new_action_metrics() -> dict:
    return {
        "action_type_tokens": 0,
        "action_type_correct": 0,
        "attach_target_tokens": 0,
        "attach_target_correct": 0,
        "attach_target_decisions": 0,
        "attach_target_exact": 0,
        "attack_action_tokens": 0,
        "attack_action_correct": 0,
        "attack_action_decisions": 0,
        "attack_action_exact": 0,
    }


def _update_action_metrics(metrics: dict, batch: dict[str, torch.Tensor], logits: torch.Tensor) -> None:
    targets = batch["targets"]
    mask = targets.ne(-100)
    predictions = logits.argmax(-1)
    option_types = batch["option_cat"][..., 0]
    option_count = option_types.size(1)
    safe_targets = targets.clamp(min=0, max=option_count - 1)
    safe_predictions = predictions.clamp(min=0, max=option_count - 1)
    rows = torch.arange(targets.size(0), device=targets.device).unsqueeze(1)
    gathered_target_types = option_types[rows, safe_targets]
    gathered_predicted_types = option_types[rows, safe_predictions]
    stop_types = torch.full_like(gathered_target_types, -1)
    target_types = torch.where(
        mask & targets.ge(0) & targets.lt(option_count),
        gathered_target_types,
        stop_types,
    )
    predicted_types = torch.where(
        predictions.lt(option_count),
        gathered_predicted_types,
        stop_types,
    )
    metrics["action_type_tokens"] += int(mask.sum().item())
    metrics["action_type_correct"] += int((target_types.eq(predicted_types) & mask).sum().item())
    exact = (predictions.eq(targets) | ~mask).all(1)
    for label, action_type in (("attach", 9), ("attack", 14)):
        selected = target_types.eq(action_type) & mask
        selected_decisions = selected.any(1)
        correct_tokens = predictions.eq(targets) & selected
        metrics[f"{label}_target_tokens" if label == "attach" else f"{label}_action_tokens"] += int(selected.sum().item())
        metrics[f"{label}_target_correct" if label == "attach" else f"{label}_action_correct"] += int(correct_tokens.sum().item())
        if label == "attach":
            metrics["attach_target_decisions"] += int(selected_decisions.sum().item())
            metrics["attach_target_exact"] += int((exact & selected_decisions).sum().item())
        else:
            metrics["attack_action_decisions"] += int(selected_decisions.sum().item())
            metrics["attack_action_exact"] += int((exact & selected_decisions).sum().item())


def _finalize_action_metrics(metrics: dict) -> dict:
    return {
        **metrics,
        "action_type_accuracy": metrics["action_type_correct"] / max(1, metrics["action_type_tokens"]),
        "attach_target_token_accuracy": metrics["attach_target_correct"] / max(1, metrics["attach_target_tokens"]),
        "attach_target_exact_action": metrics["attach_target_exact"] / max(1, metrics["attach_target_decisions"]),
        "attack_action_token_accuracy": metrics["attack_action_correct"] / max(1, metrics["attack_action_tokens"]),
        "attack_action_exact": metrics["attack_action_exact"] / max(1, metrics["attack_action_decisions"]),
    }


def _weights_from_audits(audits: list[dict], device: torch.device) -> torch.Tensor:
    return torch.tensor([_audit_sample_weight(audit) for audit in audits], dtype=torch.float32, device=device)


def save_model(path: Path, stage: int, validation: dict) -> dict:
    temporary = path.with_suffix(path.suffix + ".tmp")
    metadata = {
        "architecture": "SemanticPolicy",
        "actor_schema": fields.SCHEMA_VERSION,
        "model_config": model_config.to_dict(),
        "parameter_count": parameter_count,
        "stage": stage,
        "epoch": stage,
        "validation": validation,
        "dataset_manifest_sha256": dataset.manifest_sha256,
        "initialized_from_checkpoint": None,
        "random_initialization": True,
        "winner_only": True,
        "training_mode": "merged_recent3_cleaned_bc",
    }
    payload = {
        "schema_version": "0034_model_only_checkpoint_v1",
        "state_dict": {
            name: value.detach().cpu()
            for name, value in parallel.module.policy.named_parameters()
        },
        "metadata": metadata,
    }
    torch.save(payload, temporary)
    temporary.replace(path)
    return {
        "path": path.name,
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
        "stage": stage,
    }


def train_partition_pass(stage: int, date: str, partition_dataset) -> dict:
    parallel.train()
    torch.cuda.reset_peak_memory_stats(primary)
    source = partition_dataset.iter_audited_batches(
        "train",
        TRAIN_BATCH_SIZE,
        seed=SEED + stage,
        length_bucketed=True,
    )
    prefetch = prefetch_module.PrefetchIterator(source, depth=PREFETCH_DEPTH)
    loss_sum = 0.0
    weighted_loss_sum = 0.0
    weight_sum = 0.0
    tokens = 0
    correct = 0
    exact = 0
    decisions = 0
    updates = 0
    skipped_updates = 0
    action_metrics = _new_action_metrics()
    started = time.time()
    try:
        for batch_index, item in enumerate(prefetch, 1):
            raw, audits = item
            batch = move_batch(raw)
            weights = _weights_from_audits(audits, primary)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=amp_dtype):
                logits = parallel(batch)
                mask = batch["targets"].ne(-100)
                token_losses = F.cross_entropy(
                    logits[mask], batch["targets"][mask], reduction="none"
                )
                token_weights = weights.unsqueeze(1).expand_as(mask)[mask]
                loss = (token_losses * token_weights).sum() / token_weights.sum().clamp_min(1e-6)
                unweighted_loss = token_losses.mean()
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite 0034 weighted BC loss")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = nn.utils.clip_grad_norm_(parallel.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                skipped_updates += 1
                optimizer.zero_grad(set_to_none=True)
                scaler.update()
                continue
            scaler.step(optimizer)
            scaler.update()
            updates += 1
            token_count = int(mask.sum().item())
            matches = logits.argmax(-1).eq(batch["targets"]) & mask
            decision_count = int(batch["targets"].size(0))
            loss_sum += float(unweighted_loss.detach().cpu()) * token_count
            weighted_loss_sum += float(loss.detach().cpu()) * float(token_weights.sum().detach().cpu())
            weight_sum += float(token_weights.sum().detach().cpu())
            tokens += token_count
            correct += int(matches.sum().item())
            exact += int((matches | ~mask).all(1).sum().item())
            decisions += decision_count
            _update_action_metrics(action_metrics, batch, logits)
            if batch_index == 1 or batch_index % 100 == 0:
                print(json.dumps({
                    "event": "0034_cleaned_train_progress",
                    "stage": stage,
                    "date": date,
                    "batch": batch_index,
                    "decisions": decisions,
                    "expected_decisions": partition_dataset.split_counts["train"],
                    "loss": loss_sum / max(tokens, 1),
                    "weighted_loss": weighted_loss_sum / max(weight_sum, 1e-6),
                }, sort_keys=True), flush=True)
    finally:
        prefetch.close()
    if decisions != partition_dataset.split_counts["train"]:
        raise ValueError(f"0034 cleaned train coverage mismatch: {decisions} != {partition_dataset.split_counts['train']}")
    if not tokens or not updates:
        raise ValueError("0034 cleaned train produced no valid updates")
    return {
        "loss": loss_sum / tokens,
        "weighted_loss": weighted_loss_sum / max(weight_sum, 1e-6),
        "mean_sample_weight": weight_sum / max(tokens, 1),
        "token_accuracy": correct / tokens,
        "teacher_exact_action": exact / decisions,
        "tokens": tokens,
        "decisions": decisions,
        "updates": updates,
        "skipped_updates": skipped_updates,
        "seconds": time.time() - started,
        "cuda_peak_allocated_bytes": torch.cuda.max_memory_allocated(primary),
        "cuda_peak_reserved_bytes": torch.cuda.max_memory_reserved(primary),
        **_finalize_action_metrics(action_metrics),
    }


def validate_dataset(stage: int, label: str, eval_dataset) -> dict:
    parallel.eval()
    source = eval_dataset.iter_audited_batches(
        "validation",
        VALIDATION_BATCH_SIZE,
        seed=SEED,
        length_bucketed=False,
    )
    prefetch = prefetch_module.PrefetchIterator(source, depth=PREFETCH_DEPTH)
    loss_sum = 0.0
    weighted_loss_sum = 0.0
    weight_sum = 0.0
    tokens = 0
    correct = 0
    exact = 0
    decisions = 0
    action_metrics = _new_action_metrics()
    started = time.time()
    try:
        with torch.inference_mode():
            for raw, audits in prefetch:
                batch = move_batch(raw)
                weights = _weights_from_audits(audits, primary)
                with torch.autocast("cuda", dtype=amp_dtype):
                    logits = parallel(batch)
                    mask = batch["targets"].ne(-100)
                    token_losses = F.cross_entropy(
                        logits[mask], batch["targets"][mask], reduction="none"
                    )
                    token_weights = weights.unsqueeze(1).expand_as(mask)[mask]
                loss_sum += float(token_losses.sum().cpu())
                weighted_loss_sum += float((token_losses * token_weights).sum().cpu())
                weight_sum += float(token_weights.sum().cpu())
                tokens += int(mask.sum().item())
                matches = logits.argmax(-1).eq(batch["targets"]) & mask
                decisions += int(batch["targets"].size(0))
                correct += int(matches.sum().item())
                exact += int((matches | ~mask).all(1).sum().item())
                _update_action_metrics(action_metrics, batch, logits)
    finally:
        prefetch.close()
    if decisions != eval_dataset.split_counts["validation"]:
        raise ValueError(f"0034 cleaned validation coverage mismatch: {decisions} != {eval_dataset.split_counts['validation']}")
    if not tokens or not decisions:
        raise ValueError("0034 cleaned validation is empty")
    return {
        "label": label,
        "loss": loss_sum / tokens,
        "weighted_loss": weighted_loss_sum / max(weight_sum, 1e-6),
        "mean_sample_weight": weight_sum / max(tokens, 1),
        "token_accuracy": correct / tokens,
        "teacher_exact_action": exact / decisions,
        "tokens": tokens,
        "decisions": decisions,
        "seconds": time.time() - started,
        **_finalize_action_metrics(action_metrics),
    }


## Run three complete merged-data epochs

In [ ]:

history = []
best_validation_loss = math.inf
best_artifact = None
started = time.time()
training_config = {
    "schema_version": "0034_cleaned_bc_training_v1",
    "experiment": "0034_clean_recent3_semantic_bc",
    "dataset_interval": ["2026-08-01", "2026-08-03"],
    "dataset_manifest_sha256": dataset.manifest_sha256,
    "dataset_split_counts": dataset.split_counts,
    "partition_count": len(daily_partitions),
    "epoch_definition": "one_epoch_is_full_merged_recent3_train",
    "epochs_requested": EPOCHS,
    "winner_only": True,
    "train_passes": EPOCHS,
    "validation_passes": EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "validation_batch_size": VALIDATION_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "amp_dtype": "float16",
    "grad_scaler": True,
    "gradient_clip": 1.0,
    "data_parallel_device_ids": device_ids,
    "gpu_device_names": device_names,
    "initialized_from_checkpoint": None,
    "random_initialization": True,
    "wandb": None,
    "optimizer_state_saved": False,
    "resumable_training_state_saved": False,
    "sample_weight": {
        "field": "audit.cleaning.sample_weight",
        "loss": "per_token_weighted_mean",
        "ordinary": 1.0,
        "high_confidence_attach": 1.15,
        "low_confidence_special_energy": 0.95,
    },
}
atomic_json(OUTPUT / "training_config.json", training_config)

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_partition_pass(epoch, "combined_recent3", dataset)
    validation_metrics = validate_dataset(epoch, "merged_validation_recent3", dataset)
    artifact = save_model(OUTPUT / "last_model.pt", epoch, validation_metrics)
    if validation_metrics["weighted_loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["weighted_loss"]
        shutil.copy2(OUTPUT / "last_model.pt", OUTPUT / "best_model.pt")
        best_artifact = {
            "path": "best_model.pt",
            "sha256": sha256_file(OUTPUT / "best_model.pt"),
            "bytes": (OUTPUT / "best_model.pt").stat().st_size,
            "epoch": epoch,
            "validation_loss": best_validation_loss,
            "validation_weighted_loss": validation_metrics["weighted_loss"],
        }
    row = {
        "epoch": epoch,
        "train": train_metrics,
        "validation": validation_metrics,
        "last_artifact": artifact,
        "best_artifact": best_artifact,
        "elapsed_seconds": time.time() - started,
    }
    history.append(row)
    report = {
        "schema_version": "0034_cleaned_bc_training_report_v1",
        "state": "training",
        "experiment": "0034_clean_recent3_semantic_bc",
        "model_contract": model_contract,
        "training_config": training_config,
        "dataset_inventory": inventory,
        "history": history,
        "best_artifact": best_artifact,
    }
    atomic_json(OUTPUT / "training_report.json", report)
    atomic_json(OUTPUT / "training_status.json", {
        "state": "training",
        "epoch_completed": epoch,
        "epochs_requested": EPOCHS,
        "artifact": best_artifact,
    })
    print(json.dumps({"event": "0034_cleaned_epoch_complete", **row}, sort_keys=True), flush=True)

if len(history) != EPOCHS or best_artifact is None:
    raise RuntimeError("0034 cleaned training did not complete the requested epochs")
report["state"] = "complete"
report["epochs_completed"] = len(history)
report["full_train_passes_completed"] = len(history)
report["full_validation_passes_completed"] = len(history)
report["elapsed_seconds"] = time.time() - started
atomic_json(OUTPUT / "training_report.json", report)
atomic_json(OUTPUT / "training_status.json", {
    "state": "complete",
    "epoch_completed": len(history),
    "epochs_requested": EPOCHS,
    "artifact": best_artifact,
})
print(json.dumps({
    "event": "0034_cleaned_training_complete",
    "output": str(OUTPUT),
    "epochs_completed": len(history),
    "best_artifact": best_artifact,
    "validation": history[-1]["validation"],
}, sort_keys=True), flush=True)
